# MLX Image Classification Benchmarking

An image classification notebook for the `mlx` program

In [1]:
import os

REPO_URL = "https://github.com/ralampay/mlx.git"
BASE_DIR = "/home/ralampay/workspace/mlx/tmp"
REPO_DIR = f"/{BASE_DIR}/mlx"

if os.path.exists(REPO_DIR):
  %cd {REPO_DIR}
  !git pull
else:
  !git clone {REPO_URL} {REPO_DIR}
  %cd {REPO_DIR}

!pip install -q -r requirements.txt

/home/ralampay/workspace/mlx/tmp/mlx
Already up to date.


In [2]:
import os

DATASET_ZIP_URL = "https://alive-research.s3.ap-southeast-1.amazonaws.com/tbx11k-classification-3-class-seed-42.zip"
DATA_DIR = f"{BASE_DIR}/datasets"
ZIP_PATH = f"{DATA_DIR}/dataset-3-class-tb.zip"
DATASET_NAME = "tb-3-class-seed-42-image-classification"
SEED = 42

!mkdir -p {DATA_DIR}

if not os.path.exists(ZIP_PATH):
  !wget -O {ZIP_PATH} "{DATASET_ZIP_URL}"
else:
  print(f"Dataset already downloaded to {ZIP_PATH}")

print(f"DATA_DIR: {DATA_DIR}")

Dataset already downloaded to /home/ralampay/workspace/mlx/tmp/datasets/dataset-3-class-tb.zip
DATA_DIR: /home/ralampay/workspace/mlx/tmp/datasets


In [3]:
import os
import zipfile

EXTRACT_DIR = f"{DATA_DIR}/image-classification/"

os.makedirs(EXTRACT_DIR, exist_ok=True)

# Check if the directory is empty or if it exists with content
# This assumes that if the directory exists and is not empty, extraction has already occurred.
# A more robust check might involve checking for specific files/subdirectories expected after extraction.
if not os.path.exists(EXTRACT_DIR) or not os.listdir(EXTRACT_DIR):
  with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)
  print("Extracted to:", EXTRACT_DIR)
else:
  print(f"Data already extracted to {EXTRACT_DIR}")

print("Top-level contents:", os.listdir(EXTRACT_DIR))

train_dir = f"{EXTRACT_DIR}/tbx11k-classification-3-class-seed-42/train"

print("Found class directories in:", train_dir)
print("First 10 class names:", os.listdir(train_dir)[:10])

Data already extracted to /home/ralampay/workspace/mlx/tmp/datasets/image-classification/
Top-level contents: ['tbx11k-classification-3-class-seed-42']
Found class directories in: /home/ralampay/workspace/mlx/tmp/datasets/image-classification//tbx11k-classification-3-class-seed-42/train
First 10 class names: ['sick-non-tb', 'healthy', 'tb']


In [4]:
NUM_CLASSES = len(os.listdir(train_dir))
OUTPUT_DATASET = f"{EXTRACT_DIR}tbx11k-classification-3-class-seed-42"

print(f"Output Dataset: {OUTPUT_DATASET}")
print(f"Number of classes: {NUM_CLASSES}")

Output Dataset: /home/ralampay/workspace/mlx/tmp/datasets/image-classification/tbx11k-classification-3-class-seed-42
Number of classes: 3


In [5]:
EPOCHS = 100
DEVICE = "cuda"
BATCH_SIZE = 16

models = [
    # "efficientnet_b0",
    # "mobilenet_v3_large",
    # "drax_mobilenet_v3_large",
    # "densenet121",
    "resnet18",
    "draxnet",
    "resnet50",
    # "convnext_tiny",
    # "convnext_small",
    # "convnext_base"
]

for model_name in models:
    MODEL_NAME = model_name
    RUN_NAME = f"{SEED}-{DATASET_NAME}-{MODEL_NAME}"
    ARTIFACTS_DIR = f"{BASE_DIR}/artifacts/{RUN_NAME}"
    BENCHMARK_DIR = f"{BASE_DIR}/benchmark/{SEED}-{DATASET_NAME}-{MODEL_NAME}"
    CHECKPOINT_PATH = f"{ARTIFACTS_DIR}/{MODEL_NAME}.pth"

    print(f"\n--- Starting training for model: {MODEL_NAME} ---")
    !python -m mlx \
      --mode image_classification \
      --action train \
      --dataset "{OUTPUT_DATASET}" \
      --output "{ARTIFACTS_DIR}" \
      --model "{MODEL_NAME}" \
      --epochs "{EPOCHS}" \
      --batch-size "{BATCH_SIZE}" \
      --device "{DEVICE}" \
      --height 512 --width 512 \
      --seed "{SEED}"

    print(f"Training artifacts for {MODEL_NAME} saved to: {ARTIFACTS_DIR}")

    print(f"\n--- Starting benchmarking for model: {MODEL_NAME} ---")
    !mkdir -p "{BENCHMARK_DIR}"

    !python -m mlx \
      --mode image_classification \
      --action benchmark \
      --dataset "{OUTPUT_DATASET}" \
      --model {MODEL_NAME} \
      --model-path "{CHECKPOINT_PATH}" \
      --output "{BENCHMARK_DIR}" \
      --batch-size "{BATCH_SIZE}" \
      --device "{DEVICE}"

    print(f"Benchmark results for {MODEL_NAME} saved to: {BENCHMARK_DIR}")

    import os

    print(f"Benchmark files for {MODEL_NAME}:")
    if os.path.exists(BENCHMARK_DIR):
      for name in sorted(os.listdir(BENCHMARK_DIR)):
        print("-", name)
    else:
      print("  No benchmark directory found.")


--- Starting training for model: resnet18 ---
Using global random seed=42
╭─────────── MLX ────────────╮
│ Mode: image_classification │
│ Action: train              │
│ Model: resnet18            │
╰────────────────────────────╯
                     Configuration for resnet18 (standard)                      
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃          Parameter ┃ Value                                                   ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│             action │ train                                                   │
├────────────────────┼─────────────────────────────────────────────────────────┤
│         batch_size │ 16                                                      │
├────────────────────┼─────────────────────────────────────────────────────────┤
│            colored │ True                                                    │
├────────────────────┼───────────────────

In [6]:
import os

# Define the source directory to be zipped
source_dir = f"{BASE_DIR}/benchmark"
# Define the output zip file path
output_zip_file = f"{BASE_DIR}/{SEED}-benchmark-{DATASET_NAME}.zip"

# Ensure the old version is overridden by removing it if it exists
if os.path.exists(output_zip_file):
  !rm -f "{output_zip_file}"
  print(f"Removed existing {output_zip_file}")

# Zip the directory recursively
!zip -r "{output_zip_file}" "{source_dir}"

print(f"Successfully zipped '{source_dir}' to '{output_zip_file}'")

Removed existing /home/ralampay/workspace/mlx/tmp/42-benchmark-tb-3-class-seed-42-image-classification.zip
  adding: home/ralampay/workspace/mlx/tmp/benchmark/ (stored 0%)
  adding: home/ralampay/workspace/mlx/tmp/benchmark/200-intel-image-classification-densenet121/ (stored 0%)
  adding: home/ralampay/workspace/mlx/tmp/benchmark/200-intel-image-classification-densenet121/confusion_matrix.csv (deflated 35%)
  adding: home/ralampay/workspace/mlx/tmp/benchmark/200-intel-image-classification-densenet121/metrics.csv (deflated 30%)
  adding: home/ralampay/workspace/mlx/tmp/benchmark/200-intel-image-classification-densenet121/roc_curve.png (deflated 15%)
  adding: home/ralampay/workspace/mlx/tmp/benchmark/200-intel-image-classification-densenet121/confusion_matrix.png (deflated 16%)
  adding: home/ralampay/workspace/mlx/tmp/benchmark/200-intel-image-classification-convnext_tiny/ (stored 0%)
  adding: home/ralampay/workspace/mlx/tmp/benchmark/200-intel-image-classification-convnext_tiny/confu